# 2-1. 도구 수와 선택

선택 심화. `2`를 마친 뒤 본다.

`2`에서는 겹치는 도구 여섯을 더했을 때 무엇을 고르는지 봤다. 여기서는 도구를 70개까지 늘리며 첫 호출이 맞는 비율을 잰다.
설명의 유무, 질의에 맞는 도구만 싣는 동적 로딩, 도구 결과의 형식이 각각 그 비율을 얼마나 바꾸는지 본다.

### 공통 설정

`1`과 같은 블록이다. 측정은 대부분 하위 모델로 하고 상위 모델은 한 번만 비교한다. 동적 로딩에 임베딩 모델을 쓴다.
호출이 수백 번이라 스레드로 동시에 보낸다. 추론 effort 는 `2`와 같이 기본값으로 둔다.

In [ ]:
import json
import os
import random
import re
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(".env", usecwd=True), override=True)
MODEL = os.environ["OPENAI_DEFAULT_MODEL"]
LOWER = os.environ["OPENAI_LOWER_MODEL"]
EMBEDDING = os.environ["OPENAI_EMBEDDING_MODEL"]
client = OpenAI()

PRICES = {MODEL: (2.50, 15.00), LOWER: (0.75, 4.50), EMBEDDING: (0.02, 0.0)}
USAGE = []


def keep(resp):
    USAGE.append((resp.model, resp.usage))
    return resp

### 도구 카탈로그

정의를 찍어내는 `spec`은 `2`와 같다.

In [ ]:
def spec(name, desc, props):
    return {"type": "function", "name": name, "description": desc, "strict": True,
            "parameters": {"type": "object", "properties": props,
                           "required": list(props), "additionalProperties": False}}


def arg(name, desc):
    return {name: {"type": "string", "description": desc}}


TITLE = arg("title", "문서 제목")
PATTERN = arg("pattern", "찾을 낱말")

코퍼스 도구 열 개다. `2`의 아홉과 달리 서로 겹치지 않는다. 하는 일이 저마다 다르고 설명도 그것을 적었다.

In [ ]:
NEAR = [
    spec("grep_docs", "낱말이 들어 있는 문서를 코퍼스 전체에서 찾아 제목과 문서별 등장 횟수를 돌려준다.", PATTERN),
    spec("read_doc", "문서 한 편의 본문 앞부분을 돌려준다.", TITLE),
    spec("list_docs", "코퍼스에 있는 문서 제목을 모두 돌려준다.", {}),
    spec("count_in_doc", "문서 한 편 안에서 낱말이 몇 번 나오는지 센다.", {**TITLE, **PATTERN}),
    spec("doc_stats", "문서 한 편의 글자 수와 절 개수를 돌려준다.", TITLE),
    spec("get_section", "문서 한 편에서 지정한 절의 본문을 돌려준다.", {**TITLE, **arg("heading", "절 제목")}),
    spec("list_sections", "문서 한 편의 절 제목 목록을 돌려준다.", TITLE),
    spec("semantic_search", "질문과 뜻이 가까운 본문 조각 다섯 개를 돌려준다. 낱말이 정확히 일치하지 않아도 찾는다.",
         arg("query", "찾을 내용")),
    spec("doc_history", "문서 한 편의 수정 이력(날짜와 고친 사람)을 돌려준다.", TITLE),
    spec("related_docs", "주어진 문서와 내용이 가까운 다른 문서를 돌려준다.", TITLE),
]

코퍼스와 상관없는 범용 도구 스무 개다. 그래도 `web_search`, `read_file`, `list_files` 처럼 동사가 겹치는 것이 섞여 있다.

In [ ]:
FAR = [
    spec("weather_now", "도시의 현재 날씨를 돌려준다.", arg("city", "도시")),
    spec("exchange_rate", "두 통화의 현재 환율을 돌려준다.", {**arg("base", "기준 통화"), **arg("quote", "상대 통화")}),
    spec("stock_price", "종목의 현재 주가를 돌려준다.", arg("ticker", "종목 코드")),
    spec("send_email", "이메일을 보낸다.", {**arg("to", "받는 사람"), **arg("subject", "제목"), **arg("body", "본문")}),
    spec("create_event", "캘린더에 일정을 만든다.", {**arg("name", "일정 이름"), **arg("date", "날짜")}),
    spec("translate", "글을 다른 언어로 번역한다.", {**arg("text", "원문"), **arg("target", "목표 언어")}),
    spec("calculate", "수식을 계산한다.", arg("expression", "수식")),
    spec("date_diff", "두 날짜 사이의 일수를 센다.", {**arg("start", "시작일"), **arg("end", "종료일")}),
    spec("unit_convert", "단위를 바꾼다.", {**arg("value", "값"), **arg("src", "원래 단위"), **arg("dst", "바꿀 단위")}),
    spec("web_search", "웹을 검색해 결과 다섯 개를 돌려준다.", arg("query", "검색어")),
    spec("get_time", "시간대의 현재 시각을 돌려준다.", arg("timezone", "시간대")),
    spec("shorten_url", "긴 URL 을 짧게 만든다.", arg("url", "URL")),
    spec("run_python", "파이썬 코드를 실행하고 출력을 돌려준다.", arg("code", "코드")),
    spec("list_files", "디렉터리의 파일 목록을 돌려준다.", arg("path", "경로")),
    spec("read_file", "파일 내용을 돌려준다.", arg("path", "경로")),
    spec("write_file", "파일에 내용을 쓴다.", {**arg("path", "경로"), **arg("text", "내용")}),
    spec("geocode", "주소를 좌표로 바꾼다.", arg("address", "주소")),
    spec("route_time", "두 장소 사이의 이동 시간을 돌려준다.", {**arg("origin", "출발지"), **arg("dest", "도착지")}),
    spec("news_headlines", "주제의 최신 뉴스 제목을 돌려준다.", arg("topic", "주제")),
    spec("dictionary", "낱말의 사전 뜻을 돌려준다.", arg("word", "낱말")),
]

업무용 서비스를 붙이면 도구가 이런 식으로 는다. 서비스마다 검색, 읽기, 목록, 쓰기가 하나씩 생긴다. 열 서비스에서 마흔 개다.

In [ ]:
SAAS = []
for svc, label in [("notion", "노션"), ("confluence", "컨플루언스"), ("drive", "구글 드라이브"),
                   ("sharepoint", "셰어포인트"), ("wiki", "사내 위키")]:
    SAAS += [spec(f"{svc}_search", f"{label}에서 문서를 검색한다.", arg("query", "검색어")),
             spec(f"{svc}_read", f"{label} 문서 한 편의 내용을 돌려준다.", arg("doc_id", "문서 ID")),
             spec(f"{svc}_list", f"{label}의 문서 목록을 돌려준다.", arg("folder", "폴더")),
             spec(f"{svc}_create", f"{label}에 새 문서를 만든다.", {**arg("name", "문서 이름"), **arg("text", "내용")})]
for svc, label in [("slack", "슬랙"), ("teams", "팀즈"), ("github", "깃허브"), ("jira", "지라"), ("crm", "CRM")]:
    SAAS += [spec(f"{svc}_search", f"{label}에서 검색한다.", arg("query", "검색어")),
             spec(f"{svc}_get", f"{label}에서 항목 하나를 가져온다.", arg("item_id", "항목 ID")),
             spec(f"{svc}_post", f"{label}에 글을 올린다.", {**arg("target", "대상"), **arg("text", "내용")}),
             spec(f"{svc}_count", f"{label}에서 조건에 맞는 항목 수를 센다.", arg("query", "조건"))]

ALL = NEAR + FAR + SAAS
BY = {t["name"]: t for t in ALL}
print(len(NEAR), len(FAR), len(SAAS), "합", len(ALL))

### 질의와 정답

코퍼스 도구마다 질의 두 개를 두고 처음 불러야 할 도구를 정답으로 정한다.
첫 호출만 보고 도구는 실행하지 않는다. 그래서 도구 대부분은 구현이 없어도 된다.

In [ ]:
QUERIES = [
    ("grep_docs", "한국독립당을 언급한 문서는 모두 몇 편인가"),
    ("grep_docs", "광복군이라는 말이 나오는 문서를 모두 찾아 줘"),
    ("read_doc", "안중근 문서의 첫머리를 보여 줘"),
    ("read_doc", "김구 문서를 읽고 첫 문장을 그대로 옮겨 줘"),
    ("list_docs", "코퍼스에 어떤 문서들이 있는지 제목을 전부 보여 줘"),
    ("list_docs", "이 코퍼스의 문서 목록을 알려 줘"),
    ("count_in_doc", "김구 문서 안에서 광복군이라는 말이 몇 번 나오나"),
    ("count_in_doc", "이승만 문서에 하와이가 몇 번 나오는지 세어 줘"),
    ("doc_stats", "이승만 문서의 전체 글자 수를 알려 줘"),
    ("doc_stats", "김구 문서가 얼마나 긴지 글자 수로 알려 줘"),
    ("get_section", "안중근 문서의 '생애' 절 본문을 보여 줘"),
    ("get_section", "김구 문서에서 '평가' 절을 가져와 줘"),
    ("list_sections", "서재필 문서가 어떤 절들로 이루어져 있는지 절 제목을 알려 줘"),
    ("list_sections", "이회영 문서의 목차를 보여 줘"),
    ("semantic_search", "독립운동 자금을 어떻게 마련했는지 설명한 대목을 찾아 줘"),
    ("semantic_search", "일본 고위 인사를 저격한 일을 다룬 부분을 찾아 줘"),
    ("doc_history", "김구 문서가 마지막으로 언제 고쳐졌는지 알려 줘"),
    ("doc_history", "안중근 문서를 누가 수정해 왔는지 이력을 보여 줘"),
    ("related_docs", "안창호 문서와 내용이 비슷한 다른 문서를 찾아 줘"),
    ("related_docs", "조선민족혁명당 문서와 관련 있는 다른 문서를 알려 줘"),
]
print(len(QUERIES), "개 질의")

도구 묶음은 정답 도구 하나에 나머지를 무작위로 더해 만든다. 순서도 섞는다. 시드를 고정해 같은 조건은 같은 묶음이 된다.

In [ ]:
def toolset(target, near, far, saas, seed):
    rnd = random.Random(seed)
    others = [t for t in NEAR if t["name"] != target]
    tools = [BY[target]] + rnd.sample(others, near) + rnd.sample(FAR, far) + rnd.sample(SAAS, saas)
    rnd.shuffle(tools)
    return tools


def first_pick(query, tools, model=LOWER):
    r = keep(client.responses.create(model=model, input=query, tools=tools, store=False))
    calls = [i.name for i in r.output if i.type == "function_call"]
    return (calls[0] if calls else None), r.usage.input_tokens

질의 스무 개를 조건마다 두 번씩 돌린다. 반복마다 다른 묶음을 뽑는다. 정확도, 호출당 입력 토큰, 틀린 쌍을 돌려준다.

In [ ]:
def measure(jobs, model=LOWER):
    with ThreadPoolExecutor(8) as ex:
        res = list(ex.map(lambda j: (j[0], *first_pick(j[1], j[2], model)), jobs))
    acc = sum(t == p for t, p, _ in res) / len(res)
    tokens = sum(n for *_, n in res) / len(res)
    wrong = Counter((t, p) for t, p, _ in res if t != p)
    return acc, tokens, wrong


def sweep(label, near, far, saas, model=LOWER, reps=2, strip=False):
    jobs = []
    for i, (t, q) in enumerate(QUERIES):
        for r in range(reps):
            tools = toolset(t, near, far, saas, seed=i * 7 + r)
            if strip:
                tools = [{**x, "description": ""} for x in tools]
            jobs.append((t, q, tools))
    acc, tokens, wrong = measure(jobs, model)
    print(f"{label:16s} {model:14s} 정확도 {acc:4.0%}  입력 {tokens:5.0f} 토큰/호출  틀린 쌍 {dict(wrong)}")
    return acc, tokens

### 1. 도구 수를 늘린다

정답 하나에 다른 도구를 더해 3개, 10개, 30개, 70개로 늘린다. 10개는 코퍼스 도구 전부이고 70개는 카탈로그 전부다.

In [ ]:
SIZES = [("3개", 0, 2, 0), ("10개", 9, 0, 0), ("30개", 9, 20, 0), ("70개", 9, 20, 40)]
SCALE = {label: sweep(label, *n) for label, *n in SIZES}

틀린 쌍을 읽는다. 엉뚱한 도구보다는 `list_docs` 로 제목부터 확인하려 하거나 `web_search`, `wiki_search` 같은 다른 검색으로 새는 일이 많다.
앞의 것은 루프에서 한 턴을 더 쓰고 끝나지만 뒤의 것은 코퍼스 밖에서 답을 찾는다.
질의가 스무 개라 한두 문항 차이는 실행 편차 안이다. 조건 사이의 몇 퍼센트 차이보다 틀린 쌍의 종류를 본다.

같은 70개를 상위 모델에 준다.

In [ ]:
SCALE["70개(상위)"] = sweep("70개", 9, 20, 40, model=MODEL, reps=1)

도구 정의는 매 호출의 입력에 붙는다. 루프에서는 턴마다 붙으므로 위의 토큰에 턴 수를 곱한 것이 실제 비용이다.

### 2. 설명을 지우면

같은 묶음에서 설명만 지운다. 이름과 인자 스키마는 남는다. 이름만으로 고를 수 있는지 본다.
</br>영어로 쓰는게 더 정확도도 높긴함

In [ ]:
STRIPPED = {label: sweep(label + " 설명 없음", *n, strip=True) for label, *n in [SIZES[1], SIZES[3]]}

설명이 있을 때와 나란히 둔다.

In [ ]:
for label in STRIPPED:
    print(f"{label:4s}  설명 있음 {SCALE[label][0]:4.0%}   설명 없음 {STRIPPED[label][0]:4.0%}")

### 3. 질의에 맞는 도구만 싣는다

70개를 다 싣지 않고 질의와 가까운 몇 개만 싣는다. 도구 설명을 임베딩해 두고 질의와 내적이 큰 순서로 고른다.
`3`의 검색 도구와 같은 방식이고, 찾는 대상이 본문이 아니라 도구일 뿐이다.
</br> 길어지면 길어질 수록, 앞/뒤에 중요 도구를 두는게 맞음

In [ ]:
def embed(texts):
    r = keep(client.embeddings.create(model=EMBEDDING, input=texts))
    v = np.array([d.embedding for d in r.data])
    return v / np.linalg.norm(v, axis=1, keepdims=True)


TOOL_VEC = embed([f"{t['name']}: {t['description']}" for t in ALL])
QUERY_VEC = embed([q for _, q in QUERIES])


def retrieve(i, k, exclude=()):
    order = (TOOL_VEC @ QUERY_VEC[i]).argsort()[::-1]
    return [ALL[j] for j in order if ALL[j]["name"] not in exclude][:k]


print([t["name"] for t in retrieve(0, 5)])

`k`를 바꿔 가며 돌린다. 재현율은 정답 도구가 실린 비율이다. 실리지 않은 도구는 모델이 고를 수 없으므로 정확도의 상한이 된다.

In [ ]:
for k in [3, 5, 8]:
    jobs = [(t, q, retrieve(i, k)) for i, (t, q) in enumerate(QUERIES)]
    recall = sum(any(x["name"] == t for x in tools) for t, _, tools in jobs) / len(jobs)
    acc, tokens, wrong = measure(jobs)
    print(f"k={k}  재현율 {recall:4.0%}  정확도 {acc:4.0%}  입력 {tokens:4.0f} 토큰/호출  틀린 쌍 {dict(wrong)}")

검색이 놓치는 것은 질의에 도구의 낱말이 드러나지 않는 경우다. 자주 쓰는 도구 몇 개는 늘 싣고, 나머지만 검색으로 붙인다.

In [ ]:
CORE = ["grep_docs", "read_doc", "semantic_search"]
jobs = [(t, q, [BY[c] for c in CORE] + retrieve(i, 5, exclude=CORE)) for i, (t, q) in enumerate(QUERIES)]
recall = sum(any(x["name"] == t for x in tools) for t, _, tools in jobs) / len(jobs)
acc, tokens, wrong = measure(jobs)
print(f"고정 3 + 검색 5  재현율 {recall:4.0%}  정확도 {acc:4.0%}  입력 {tokens:4.0f} 토큰/호출  틀린 쌍 {dict(wrong)}")

### 4. 결과 형식이 답을 바꾼다

도구를 맞게 골라도 결과를 잘못 읽으면 답이 틀린다. `grep_docs` 하나로 같은 질문을 하고 결과 형식만 바꾼다.
JSON 목록, 한 줄에 하나씩 쓴 텍스트, 문서 수를 미리 세고 횟수 순으로 정렬한 JSON 셋이다.

In [ ]:
CORPUS = Path("../../data/corpus/graph")
DOCS = {f.stem: f.read_text(encoding="utf-8") for f in sorted(CORPUS.glob("*.md"))}


def hits(pattern):
    return [{"title": t, "count": text.count(pattern)} for t, text in DOCS.items() if pattern in text]


def as_json(p):
    return json.dumps(hits(p), ensure_ascii=False)


def as_text(p):
    return "\n".join(f"{h['title']} {h['count']}" for h in hits(p))


def as_counted(p):
    h = sorted(hits(p), key=lambda d: -d["count"])
    return json.dumps({"문서수": len(h), "문서": h}, ensure_ascii=False)


FORMATS = {"JSON": as_json, "텍스트": as_text, "미리 센 값": as_counted}
print(as_counted("하얼빈")[:120])

문서가 스물에서 서른여섯 편 나오는 낱말 여섯 개로 세 가지를 묻는다. 정답은 같은 `hits`에서 만든다.

In [ ]:
WORDS = ["상하이", "미국", "만주", "이승만", "여운형", "총독부"]
QUESTIONS = {"문서 수": "'{w}'이 나오는 문서는 몇 편인가. 숫자만 답한다.",
             "최다 문서": "'{w}'이 가장 많이 나오는 문서의 제목만 답한다.",
             "5회 이상": "'{w}'이 5번 이상 나오는 문서는 몇 편인가. 숫자만 답한다."}


def truth(w, kind):
    h = hits(w)
    if kind == "문서 수":
        return str(len(h))
    if kind == "최다 문서":
        return max(h, key=lambda d: d["count"])["title"]
    return str(sum(d["count"] >= 5 for d in h))


GREP = [NEAR[0]]
print({w: [truth(w, k) for k in QUESTIONS] for w in WORDS})

한 바퀴짜리 루프다. 모델이 `grep_docs`를 부르면 정한 형식으로 결과를 돌려주고 답을 받는다.

In [ ]:
def ask(fmt, w, kind):
    q = QUESTIONS[kind].format(w=w)
    r = keep(client.responses.create(model=LOWER, input=q, tools=GREP, store=False))
    messages = [{"role": "user", "content": q}] + [i.model_dump(exclude_none=True) for i in r.output]
    calls = [i for i in r.output if i.type == "function_call"]
    for c in calls:
        messages.append({"type": "function_call_output", "call_id": c.call_id,
                         "output": FORMATS[fmt](json.loads(c.arguments)["pattern"])})
    if calls:
        r = keep(client.responses.create(model=LOWER, input=messages, tools=GREP, store=False))
    answer, t = r.output_text, truth(w, kind)
    return t in answer if kind == "최다 문서" else bool(re.search(rf"(?<!\d){t}(?!\d)", answer))

형식마다 열여덟 문항을 돌린다.

In [ ]:
jobs = [(f, w, k) for f in FORMATS for w in WORDS for k in QUESTIONS]
with ThreadPoolExecutor(8) as ex:
    oks = list(ex.map(lambda j: ask(*j), jobs))

for f in FORMATS:
    row = {k: sum(ok for (ff, _, kk), ok in zip(jobs, oks) if ff == f and kk == k) for k in QUESTIONS}
    print(f"{f:6s}  " + "   ".join(f"{k} {n}/{len(WORDS)}" for k, n in row.items()))

제목 하나를 고르는 것은 형식과 거의 상관없다. 세는 질문은 형식에 따라 갈린다.
목록이 길면 모델은 세지 않고 어림한다. 텍스트 나열에서는 문서 수조차 맞히지 못한다. 셀 수 있는 것은 도구가 세서 돌려준다.
"5회 이상" 은 정렬해 둔 목록에서도 다 맞지는 않는다. 자주 묻는 집계라면 그 값 자체를 돌려주는 인자를 도구에 둔다.

### 정리

| 무엇을 바꿨나 | 정확도 | 비용 |
|---|---|---|
| 도구 수 | 설명이 좋으면 천천히 떨어진다. 상위 모델은 70개에서도 거의 맞는다 | 도구 수에 비례해 매 호출, 매 턴 붙는다 |
| 설명 | 지우면 같은 수에서 크게 떨어진다. 도구가 많을수록 더 떨어진다 | 조금 준다 |
| 동적 로딩 | 검색이 놓친 도구는 고를 수 없다. 늘 쓰는 도구를 고정하면 메워진다 | 크게 준다 |
| 결과 형식 | 세는 질문에서 크게 갈린다 | 거의 같다 |

도구가 늘어서 생기는 문제는 고르는 능력보다 비용 쪽에 먼저 온다. 도구를 줄이기 어렵다면 설명을 고치고, 싣는 도구를 질의마다 고르고, 결과는 모델이 셀 필요 없게 만든다.

### 비용 확인

In [ ]:
total = 0.0
for name, u in USAGE:
    p = next((PRICES[k] for k in sorted(PRICES, key=len, reverse=True) if name.startswith(k)), None)
    if p:
        total += getattr(u, "input_tokens", getattr(u, "prompt_tokens", 0)) / 1e6 * p[0]
        total += getattr(u, "output_tokens", 0) / 1e6 * p[1]

print(f"호출 {len(USAGE)}회")
print(f"입력 {sum(getattr(u, 'input_tokens', getattr(u, 'prompt_tokens', 0)) for _, u in USAGE):,} 토큰"
      f"   출력 {sum(getattr(u, 'output_tokens', 0) for _, u in USAGE):,} 토큰")
print(f"누적 ${total:.4f}")